In [2]:
import polars as pl
import gemmi, math
from collections import defaultdict
from procompa import get_project_root

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

In [3]:
CPX_1861 = data_dir/"Pipeline/10_all_CP_complexes/CombFold/P31539x6_P38276x2_Q12118x2_Q12125x2_Q12285x2_pool_output/assembled_results/output_clustered_1.pdb"

st = gemmi.read_structure(str(CPX_1861))
st.setup_entities()
model = st[0]

rows = []
for ch in model:
    for res in ch:
        ca = res.find_atom("CA", "*")
        if ca is None:
            continue
        x, y, z = ca.pos.x, ca.pos.y, ca.pos.z
        rows.append({
            "chain": ch.name,
            "resnum": res.seqid.num,
            "resname": res.name,
            "x": x, "y": y, "z": z,
            "is_nan": math.isnan(x) or math.isnan(y) or math.isnan(z),
        })

ca = pl.DataFrame(rows)

In [4]:
chain_list_path = data_dir/"Pipeline/10_all_CP_complexes/CombFold/P31539x6_P38276x2_Q12118x2_Q12125x2_Q12285x2_pool_output/_unified_representation/assembly_output/chain.list"


uni2chains = defaultdict(list)
for line in open(chain_list_path):
    line = line.strip()
    if not line:
        continue
    u, c = line.removesuffix(".pdb").split("_")
    uni2chains[u].append(c)

chain2uni = {c: u for u, cs in uni2chains.items() for c in cs}
print(dict(uni2chains))

{'P31539': ['A', 'B', 'C', 'D', 'E', 'F'], 'P38276': ['G', 'H'], 'Q12118': ['I', 'J'], 'Q12125': ['K', 'L'], 'Q12285': ['M', 'N']}


In [5]:
Xlinks = pl.read_parquet(data_dir/"Xlinks/combined_xlink.parquet")

In [6]:
import itertools
from collections import defaultdict

THRESH = 30.0

# CA lookup, dropping NaN-coordinate chains
ca_xyz, resname = {}, {}
for ch in model:
    for res in ch:
        a = res.find_atom("CA", "*")
        if a is None or math.isnan(a.pos.x):
            continue
        ca_xyz[(ch.name, res.seqid.num)] = a.pos
        resname[(ch.name, res.seqid.num)] = res.name

prots = set(uni2chains)
xl = (Xlinks
      .filter(pl.col("uniprot_id1").is_in(prots) & pl.col("uniprot_id2").is_in(prots))
      .unique(["uniprot_id1", "uniprot_id2", "pos1", "pos2"]))

rows = []
for r in xl.iter_rows(named=True):
    u1, u2, p1, p2 = r["uniprot_id1"], r["uniprot_id2"], r["pos1"], r["pos2"]
    best, best_intra = None, None
    for c1, c2 in itertools.product(uni2chains[u1], uni2chains[u2]):
        if c1 == c2 and p1 == p2:            # degenerate self-pair
            continue
        a, b = ca_xyz.get((c1, p1)), ca_xyz.get((c2, p2))
        if a is None or b is None:
            continue
        d = a.dist(b)
        if best is None or d < best[0]:
            best = (d, c1, c2)
        if c1 == c2 and (best_intra is None or d < best_intra):
            best_intra = d
    rows.append({**r,
        "res1": resname.get((uni2chains[u1][0], p1)),
        "res2": resname.get((uni2chains[u2][0], p2)),
        "min_dist":    None if best is None else best[0],
        "chain1":      None if best is None else best[1],
        "chain2":      None if best is None else best[2],
        "intra_chain": None if best is None else best[1] == best[2],
        "min_intra":   best_intra,
        "satisfied":   None if best is None else best[0] <= THRESH,
    })

res = pl.DataFrame(rows).sort("min_dist", nulls_last=True)

In [ ]:
# crosslinks also under:"C:\Users\damme\Downloads\crosslink_viewer (1).html"